<div style="background: linear-gradient(135deg, #1a1a2e 0%, #16213e 50%, #0f3460 100%); padding: 40px; border-radius: 15px; text-align: center; margin-bottom: 20px;">
<h1 style="color: #e94560; font-size: 2.5em; margin-bottom: 10px;"> Visão Computacional Aplicada</h1>
<h2 style="color: #a8dadc; font-size: 1.5em; font-weight: 300;">Machine Learning para Reconhecimento de Imagens e Objetos</h2>
<p style="color: #8ecae6; font-size: 1.1em; margin-top: 15px;">MBA em Inteligência Artificial — FIAP</p>
<hr style="border-color: #e94560; margin: 20px 0;">
<p style="color: #caf0f8;"> <em>Execute em ambiente com GPU (Google Colab recomendado)</em></p>
</div>

## Roteiro da Aula

| # | Módulo | Técnica Principal | Aplicação de Negócio |
|---|--------|------------------|----------------------|
| 1 | **Calibração de Câmeras** | Geometria Epipolar (OpenCV) | Inspeção industrial, visão estéreo |
| 2 | **Estimativa de Profundidade** | MiDaS — Monocular Depth | Robótica, AR, veículos autônomos |
| 3 | **Deep Learning — Classificação** | CNN do zero (Keras) | Reconhecimento de objetos COIL-100 |
| 4 | **Transfer Learning** | VGG19 fine-tuning | Linguagem de sinais (ASL) |
| 5 | **Detecção de Objetos** | YOLO12 (Ultralytics) | Detecção em tempo real + modelo customizado |
| 6 | **Segmentação** | RF-DETR + SAM (Meta) | Segmentação de instâncias |
| 7 | **Vision Transformer** | ViT + Attention Rollout | Classificação médica interpretável |

---

### Como usar este notebook

> **Leia** os textos explicativos antes de executar cada célula.
> **Execute** célula por célula de cima para baixo (Shift+Enter).
> **Reflita** sobre os resultados: o que o modelo aprendeu? Onde ele errou?
> **Experimente** as células marcadas com — são exercícios práticos para você completar.


---
## Setup do Ambiente

A primeira etapa é clonar o repositório e instalar as dependências.

> **Por que GPU?** Redes neurais convolucionais realizam bilhões de multiplicações de matrizes por segundo. Uma GPU com milhares de núcleos CUDA executa essas operações em paralelo — o que levaria horas na CPU leva minutos na GPU.


In [ ]:
# Clona o repositório com os materiais da aula
!rm -rf fiap-ml-visao-computacional/
!git clone https://github.com/FIAPON/fiap-ml-visao-computacional.git
%cd fiap-ml-visao-computacional/aula-5-machine-learning-aplicado/


In [ ]:
# Instalação das bibliotecas principais
!pip install timm watermark --quiet


In [ ]:
# Importações globais utilizadas ao longo de todo o notebook
from IPython.display import HTML
from google.colab.output import eval_js
from base64 import b64decode, b64encode
import io
from PIL import Image
import cv2
import torch
import matplotlib.pyplot as plt
import numpy as np
import copy as cp
import os

print("Ambiente configurado!")
print(f" Device: {'GPU ' if torch.cuda.is_available() else 'CPU (considere ativar GPU no Colab)'}")


In [ ]:
import watermark
%reload_ext watermark
%watermark -a "MBA FIAP — Visão Computacional" --iversions


---
# Módulo 1 — Calibração de Câmeras

## Contexto de Negócio

Câmeras do mundo real introduzem **distorções** — objetos retos aparecem curvados, especialmente nas bordas. Isso é crítico em:
- **Visão industrial**: medição de peças e inspeção de qualidade
- **Veículos autônomos**: estimativa de distância e detecção de faixas
- **Realidade aumentada**: sobreposição correta de elementos virtuais

## Como funciona a calibração?

```
 Foto do tabuleiro de xadrez (padrão conhecido)
 ↓
 Detectar cantos internos → cv2.findChessboardCorners()
 ↓
 Estimar parâmetros intrínsecos → cv2.calibrateCamera()
 ↓
 Aplicar correção (undistortion) em vídeo
```

### Parâmetros estimados

| Parâmetro | O que representa |
|-----------|-----------------|
| `mtx` | Matriz da câmera: distância focal (fx, fy) e ponto principal (cx, cy) |
| `dist` | Coeficientes de distorção radial (k1, k2, k3) e tangencial (p1, p2) |
| **Erro RMS** | Erro médio de reprojeção em pixels — **< 1px = excelente** |


---
## 1.0 O Problema — Vendo a Distorção com Seus Próprios Olhos

Antes de calibrar, precisamos entender **o que estamos corrigindo**.

Toda câmera real usa uma lente física. Lentes introduzem **distorção geométrica** — linhas que são retas no mundo real aparecem curvas na imagem. Existem dois tipos principais:

| Tipo | Causa | Efeito Visual | Exemplo Real |
|------|-------|---------------|--------------|
| **Barrel (barril)** 🍺 | Lentes grande-angulares, webcams, câmeras de segurança | Bordas curvam para *fora* — imagem "incha" | GoPro, câmera de carro |
| **Pincushion (almofada)** 📌 | Teleobjetivas, zooms com focal longa | Bordas curvam para *dentro* — imagem "encolhe" | Câmeras com zoom óptico |

> **Execute a célula abaixo** para visualizar ambas as distorções aplicadas a uma grade perfeita de linhas retas.


In [ ]:
# ─── 1.0 Visualizando Distorção Sintética ──────────────────────────────────
# Geramos uma grade de linhas RETAS e aplicamos distorção matematicamente,
# para que você veja O PROBLEMA antes de ver a solução.
import cv2
import numpy as np
import matplotlib.pyplot as plt

SIZE = 500

# Grade perfeita (representa o "mundo real")
img_grade = np.ones((SIZE, SIZE, 3), dtype=np.uint8) * 30
for i in range(0, SIZE, 50):
    cv2.line(img_grade, (i, 0),    (i, SIZE),  (180, 180, 180), 1)
    cv2.line(img_grade, (0, i),    (SIZE, i),  (180, 180, 180), 1)
# Linhas centrais em verde — facilitam ver a curvatura
cv2.line(img_grade, (SIZE // 2, 0),    (SIZE // 2, SIZE), (80, 220, 80), 2)
cv2.line(img_grade, (0, SIZE // 2),    (SIZE, SIZE // 2), (80, 220, 80), 2)

# Matriz intrínseca sintética — câmera fictícia centralizada
K_fake = np.array([[SIZE * 0.9,         0, SIZE / 2],
                   [        0, SIZE * 0.9, SIZE / 2],
                   [        0,         0,        1 ]], dtype=np.float64)

# Coeficientes: [k1, k2, p1, p2, k3]  →  k1 < 0 = barrel  |  k1 > 0 = pincushion
cenarios = [
    (np.zeros(5),
     'Grade Original\n(sem distorção)',       '#4CAF50', 'Referência'),
    (np.array([-0.45,  0.15, 0.0,  0.0, 0.0]),
     'Barrel 🍺\nk1 = -0.45',               '#2196F3', 'Webcam / GoPro / câmera de carro'),
    (np.array([ 0.45, -0.12, 0.0,  0.0, 0.0]),
     'Pincushion 📌\nk1 = +0.45',           '#FF5722', 'Teleobjetiva / zoom óptico'),
    (np.array([-0.55,  0.30, 0.008, -0.005, -0.06]),
     'Distorção Mista\n(radial + tangencial)', '#9C27B0', 'Câmera de segurança barata'),
]

fig, axes = plt.subplots(1, 4, figsize=(20, 5))
fig.patch.set_facecolor('#1a1a2e')

for ax, (dist_coef, titulo, cor, exemplo) in zip(axes, cenarios):
    m1, m2 = cv2.initUndistortRectifyMap(
        K_fake, dist_coef, None, K_fake, (SIZE, SIZE), cv2.CV_32FC1)
    distorted = cv2.remap(img_grade, m1, m2, cv2.INTER_LINEAR)
    ax.imshow(cv2.cvtColor(distorted, cv2.COLOR_BGR2RGB))
    ax.set_title(titulo, fontsize=11, color='white', pad=8)
    ax.set_facecolor('#0f3460')
    ax.tick_params(colors='white')
    for spine in ax.spines.values():
        spine.set_edgecolor(cor)
        spine.set_linewidth(2.5)
    ax.text(0.5, -0.08, f'Ex: {exemplo}', transform=ax.transAxes,
            ha='center', fontsize=8.5, color='#aaaaaa', style='italic')

plt.suptitle(
    'Tipos de distorção de lente — aplicados a uma grade de linhas retas',
    fontsize=14, color='white', fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print('\n🔍 Observe: as linhas que DEVERIAM ser retas estão curvas!')
print('   A calibração de câmera resolve exatamente isso.')


---
## 1.1 A Câmera Pinhole em 2 Minutos — O Modelo Matemático

Para *corrigir* uma câmera, precisamos entender como ela **projeta o mundo 3D em pixels 2D**.

O modelo pinhole descreve essa projeção com dois conjuntos de parâmetros:

### Parâmetros Intrínsecos — a "personalidade" da câmera
São fixos para cada câmera/lente e representam as propriedades ópticas do sensor:

```
       Ponto no mundo 3D  (X, Y, Z)
                ↓
    ┌──────────────────────────────┐
    │   Matriz Intrínseca  K       │
    │                              │
    │   K = │ fx   0   cx │        │
    │       │  0  fy   cy │        │
    │       │  0   0    1 │        │
    │                              │
    │  fx, fy : distância focal    │
    │           (em pixels)        │
    │  cx, cy : ponto principal    │
    │           (centro óptico     │
    │           projetado no sensor│
    └──────────────────────────────┘
                ↓
          Pixel  (u, v)
```

### Coeficientes de Distorção — os "defeitos" da lente

| Coeficiente | Tipo | O que faz |
|-------------|------|-----------|
| `k1, k2, k3` | Radial | Curvatura barrel/pincushion. **k1 é o mais importante.** |
| `p1, p2` | Tangencial | Desalinhamento físico entre lente e sensor |

### Parâmetros Extrínsecos — a "pose" da câmera no espaço
`rvecs` (vetores de rotação) e `tvecs` (vetores de translação) descrevem **onde a câmera está em relação ao tabuleiro** — mudam a cada foto/pose.

> **Insight chave:** a calibração estima `K` e `dist` **uma única vez**. Depois, você aplica esses parâmetros para corrigir qualquer frame capturado por aquela câmera.


---
## 1.2 Calibração com OpenCV — Passo a Passo Comentado

O padrão mais comum é o **tabuleiro de xadrez**: geometria perfeitamente conhecida, alta precisão de detecção de cantos.

### Por que tabuleiro de xadrez?
- Cantos são detectáveis com precisão **sub-pixel** (`cornerSubPix`)
- A geometria é perfeitamente conhecida (quadrados iguais, Z = 0)
- Alto contraste preto/branco garante detecção robusta

### Pipeline de 4 etapas:

```
 ① Definir coordenadas 3D reais (plano Z=0 do tabuleiro)
    objp = [(0,0,0), (1,0,0), (2,0,0), ...]
    ↓
 ② Detectar cantos 2D na imagem
    ret, corners = cv2.findChessboardCorners()
    ↓
 ③ Refinar para sub-pixel (precisão de frações de pixel!)
    corners2 = cv2.cornerSubPix()
    ↓
 ④ Resolver sistema de equações (mínimos quadrados)
    rms, K, dist, rvecs, tvecs = cv2.calibrateCamera()
```

> **Nota prática:** Em produção, use **15–30 imagens** com o tabuleiro em diferentes poses, distâncias e inclinações. Quanto mais diversidade de poses, mais robusto o modelo estimado. Aqui usamos 1 imagem por simplicidade pedagógica.


In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

video_input = 'videos/xadrez.mp4'
video_output = 'video_comparativo.mp4'
imagem_calib = 'imagens/xadrez.jpg'

# Definição do padrão de calibração 
pattern_size = (7, 7) # cantos internos do tabuleiro
n_pontos = pattern_size[0] * pattern_size[1]

# Coordenadas do mundo real (plano Z=0)
objp = np.zeros((n_pontos, 3), np.float32)
objp[:, :2] = np.mgrid[0:pattern_size[0], 0:pattern_size[1]].T.reshape(-1, 2)

# Detecção dos cantos 
img_calib = cv2.imread(imagem_calib)
gray_calib = cv2.cvtColor(img_calib, cv2.COLOR_BGR2GRAY)
ret, corners = cv2.findChessboardCorners(gray_calib, pattern_size, None)

# Refinamento sub-pixel
criteria = (cv2.TERM_CRITERIA_EPS + cv2.TERM_CRITERIA_MAX_ITER, 30, 0.001)
corners2 = cv2.cornerSubPix(gray_calib, corners, (11, 11), (-1, -1), criteria)

# Calibração matemática 
ret_rms, mtx, dist, rvecs, tvecs = cv2.calibrateCamera(
    [objp], [corners2], gray_calib.shape[::-1], None, None)

print(f" Erro RMS de reprojeção: {ret_rms:.4f} pixels")
print(f" (< 1.0 px é considerado excelente)")

# Visualização 
imgp_proj, _ = cv2.projectPoints(objp, rvecs[0], tvecs[0], mtx, dist)
erros = np.linalg.norm(corners2.reshape(-1,2) - imgp_proj.reshape(-1,2), axis=1)
errors_grid = erros.reshape(pattern_size)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

img_vis = img_calib.copy()
cv2.drawChessboardCorners(img_vis, pattern_size, corners2, ret)
axes[0].imshow(cv2.cvtColor(img_vis, cv2.COLOR_BGR2RGB))
axes[0].set_title("Cantos detectados no tabuleiro", fontsize=13)
axes[0].axis('off')

im = axes[1].imshow(errors_grid, cmap='RdYlGn_r')
plt.colorbar(im, ax=axes[1]).set_label('Erro de reprojeção (pixels)')
axes[1].set_title(f"Mapa de erro por canto\nErro médio: {erros.mean():.4f} px", fontsize=13)
plt.tight_layout()
plt.show()


---
## 1.3 Decodificando os Resultados — O que a câmera nos contou?

Após a calibração, temos números — mas o que eles **significam**?

A célula abaixo interpreta cada parâmetro em linguagem humana e gera dois mapas visuais:
- **Campo vetorial**: mostra a *direção e magnitude* do deslocamento de cada pixel
- **Mapa de calor**: evidencia que a distorção é maior nas **bordas** da imagem


In [ ]:
# ─── 1.3 Interpretação dos Parâmetros de Calibração ────────────────────────

# Extrair componentes
fx, fy   = mtx[0, 0], mtx[1, 1]
cx, cy   = mtx[0, 2], mtx[1, 2]
k1, k2, p1, p2, k3 = dist.ravel()
centro_x, centro_y = w_img / 2, h_img / 2
delta_cx = cx - centro_x
delta_cy = cy - centro_y
desal    = (delta_cx**2 + delta_cy**2) ** 0.5

sep = '─' * 60

print(f'\n📷  RELATÓRIO DE CALIBRAÇÃO DA CÂMERA')
print(sep)
print(f'  Resolução da imagem:         {w_img} × {h_img} px')
print(f'  Erro RMS de reprojeção:      {ret_rms:.4f} px', end='')
if   ret_rms < 0.5: print('  ✅ EXCELENTE')
elif ret_rms < 1.0: print('  ✅ BOM')
elif ret_rms < 2.0: print('  ⚠️  ACEITÁVEL (considere mais imagens)')
else:               print('  ❌ RUIM — recalibre com mais poses')

print(f'\n🔬  PARÂMETROS INTRÍNSECOS (Matriz K)')
print(sep)
print(f'  fx (dist. focal horizontal): {fx:>10.2f} px')
print(f'  fy (dist. focal vertical):   {fy:>10.2f} px')
ratio = fy / fx
print(f'  Razão fy/fx (deve ser ≈1.0): {ratio:>10.4f}', end='')
print('  ✅' if abs(ratio - 1) < 0.05 else '  ⚠️  pixels não-quadrados')
print(f'  cx (ponto principal x):      {cx:>10.2f} px')
print(f'  cy (ponto principal y):      {cy:>10.2f} px')
print(f'  Centro geométrico:           ({centro_x:.1f}, {centro_y:.1f}) px')
print(f'  Desalinhamento óptico:       Δx={delta_cx:+.1f}  Δy={delta_cy:+.1f}  '
      f'({desal:.1f} px total)', end='')
print('  ✅' if desal < 50 else '  ⚠️  desalinhamento elevado')

print(f'\n🌀  DISTORÇÃO DA LENTE')
print(sep)
tipo_k1 = 'Barrel 🍺 (grande-angular)' if k1 < 0 else 'Pincushion 📌 (teleobjetiva)'
print(f'  k1 = {k1:+.6f}  → {tipo_k1}')
print(f'  k2 = {k2:+.6f}  (radial 2ª ordem)')
print(f'  k3 = {k3:+.6f}  (radial 3ª ordem)')
print(f'  p1 = {p1:+.6f}  (tangencial)')
print(f'  p2 = {p2:+.6f}  (tangencial)')
total_dist = abs(k1) + abs(k2) + abs(p1) + abs(p2)
print(f'\n  Intensidade de distorção: {total_dist:.4f}', end='')
if   total_dist < 0.1: print('  ✅ Baixa — câmera de qualidade')
elif total_dist < 0.5: print('  ⚠️  Moderada — calibração importante')
else:                  print('  ❌ Alta — corrija SEMPRE antes de medir')
print(f'\n{sep}\n')

# ── Visualizações: campo vetorial + mapa de calor ────────────────────────────
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.patch.set_facecolor('#1a1a2e')

step = 40
ys, xs = np.mgrid[step // 2:h_img:step, step // 2:w_img:step]
pts = np.stack([xs.ravel(), ys.ravel()], axis=1).astype(np.float32).reshape(-1, 1, 2)
pts_ud = cv2.undistortPoints(pts, mtx, dist, P=mtx).reshape(-1, 2)
dx  = pts_ud[:, 0] - pts[:, 0, 0]
dy  = pts_ud[:, 1] - pts[:, 0, 1]
mag = np.sqrt(dx**2 + dy**2)

# Campo vetorial
ax = axes[0]
ax.set_facecolor('#0f3460')
sc = ax.quiver(pts[:, 0, 0], pts[:, 0, 1], dx, dy, mag,
               cmap='plasma', scale=200, width=0.003)
cb = plt.colorbar(sc, ax=ax)
cb.set_label('Deslocamento (px)', color='white')
cb.ax.yaxis.set_tick_params(color='white')
plt.setp(cb.ax.yaxis.get_ticklabels(), color='white')
ax.set_xlim(0, w_img); ax.set_ylim(h_img, 0)
ax.set_title('Campo Vetorial de Distorção\n(onde cada pixel se move ao ser corrigido)',
             color='white', fontsize=11)
ax.tick_params(colors='white')
ax.set_xlabel('x (px)', color='white'); ax.set_ylabel('y (px)', color='white')

# Mapa de calor
ax2 = axes[1]
ax2.set_facecolor('#0f3460')
sc2 = ax2.scatter(pts[:, 0, 0], pts[:, 0, 1], c=mag, cmap='RdYlGn_r', s=80, vmin=0)
cb2 = plt.colorbar(sc2, ax=ax2)
cb2.set_label('Magnitude (px)', color='white')
cb2.ax.yaxis.set_tick_params(color='white')
plt.setp(cb2.ax.yaxis.get_ticklabels(), color='white')
ax2.set_xlim(0, w_img); ax2.set_ylim(h_img, 0)
ax2.set_title('Mapa de Calor da Distorção\n(vermelho = maior erro nas bordas)',
              color='white', fontsize=11)
ax2.tick_params(colors='white')
ax2.set_xlabel('x (px)', color='white'); ax2.set_ylabel('y (px)', color='white')

plt.suptitle(
    'Análise Geométrica — como os pixels se movem ao ser corrigidos',
    color='white', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()


In [ ]:
# Gerar video comparativo: distorcido vs. corrigido
w_img, h_img = gray_calib.shape[1], gray_calib.shape[0]
new_mtx, roi = cv2.getOptimalNewCameraMatrix(mtx, dist, (w_img, h_img), 1, (w_img, h_img))
mapx, mapy   = cv2.initUndistortRectifyMap(mtx, dist, None, new_mtx, (w_img, h_img), cv2.CV_32FC1)

cap = cv2.VideoCapture(video_input)
if cap.isOpened():
    w   = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    h   = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps = cap.get(cv2.CAP_PROP_FPS)
    mapx2, mapy2 = cv2.initUndistortRectifyMap(mtx, dist, None, new_mtx, (w, h), cv2.CV_32FC1)

    out = cv2.VideoWriter(video_output, cv2.VideoWriter_fourcc(*'mp4v'), fps, (w * 2, h))
    while cap.isOpened():
        ok, frame = cap.read()
        if not ok:
            break
        dst = cv2.remap(frame, mapx2, mapy2, cv2.INTER_LINEAR)
        cv2.putText(frame, "ORIGINAL",  (20, 50), cv2.FONT_HERSHEY_SIMPLEX, 1.2, (0, 0, 255), 3)
        cv2.putText(dst,   "CORRIGIDO", (20, 50), cv2.FONT_HERSHEY_SIMPLEX, 1.2, (0, 200, 0), 3)
        out.write(np.hstack((frame, dst)))
    cap.release()
    out.release()

    !ffmpeg -i {video_output} -vcodec libx264 -acodec aac video_comparativo_final.mp4 -y -loglevel quiet
    print("Video comparativo gerado!")
else:
    print("Video nao encontrado:", video_input)

> ** O que observar no vídeo?**
>
> **Lado esquerdo (original):** linhas retas do tabuleiro aparecem levemente curvas nas bordas — distorção radial da lente.
> **Lado direito (corrigido):** linhas ficam perfeitamente retas após aplicar os coeficientes estimados.
> Isso é crítico para qualquer sistema de medição ou detecção baseado em câmera.


---
## 1.4 Exercício — Qual câmera você escolheria?

**Cenário real de negócio:** Você é engenheiro de visão computacional em uma linha de inspeção industrial. Seu cliente precisa medir peças com precisão de ±0,1 mm. Você recebeu **3 câmeras** para avaliar.

A célula abaixo visualiza o efeito de distorção de cada câmera (linha superior) e o resultado após calibração e correção (linha inferior).

**Analise os resultados e responda:**
1. Qual câmera apresenta menor distorção *antes* da calibração?
2. Qual **não deve** ser usada sem calibração em aplicações de medição?
3. Após calibração, as três câmeras entregam resultados equivalentes?

> 💡 **Dica:** observe especialmente as bordas da imagem e a uniformidade da correção na linha inferior.


In [ ]:
# ─── 1.4 Exercício: Comparando 3 câmeras de mercado ────────────────────────

import cv2, numpy as np, matplotlib.pyplot as plt

SZ = 420
K_ex = np.array([[SZ * 0.9,       0, SZ / 2],
                 [       0, SZ * 0.9, SZ / 2],
                 [       0,       0,       1]], dtype=np.float64)

# Grade de referência (linhas perfeitamente retas)
grade = np.ones((SZ, SZ, 3), dtype=np.uint8) * 25
for i in range(0, SZ, 42):
    cv2.line(grade, (i, 0),   (i, SZ),  (170, 170, 170), 1)
    cv2.line(grade, (0, i),   (SZ, i),  (170, 170, 170), 1)
cv2.line(grade, (SZ // 2, 0),   (SZ // 2, SZ), (80, 220, 80), 2)
cv2.line(grade, (0, SZ // 2),   (SZ, SZ // 2), (80, 220, 80), 2)

# 3 câmeras com perfis de distorção distintos
cameras = [
    dict(nome='Câmera A\nWebcam industrial USB',
         dist=np.array([-0.38, 0.14, 0.001, -0.001, 0.0]),
         desc='k1 = −0.38  (barrel moderado)',
         cor='#2196F3'),
    dict(nome='Câmera B\nCâmera industrial de precisão',
         dist=np.array([-0.04, 0.005, 0.0001, -0.0002, 0.0]),
         desc='k1 = −0.04  (distorção mínima)',
         cor='#4CAF50'),
    dict(nome='Câmera C\nCâmera de segurança barata',
         dist=np.array([-0.58, 0.35, 0.006, -0.004, -0.08]),
         desc='k1 = −0.58 + p1/p2 altos (severa)',
         cor='#FF5722'),
]

fig, axes = plt.subplots(2, 3, figsize=(16, 10))
fig.patch.set_facecolor('#1a1a2e')

for col, cam in enumerate(cameras):
    d = cam['dist']

    # Linha 0 — imagem com distorção (sem calibração)
    m1, m2    = cv2.initUndistortRectifyMap(
        K_ex, d, None, K_ex, (SZ, SZ), cv2.CV_32FC1)
    distorted = cv2.remap(grade, m1, m2, cv2.INTER_LINEAR)

    axes[0, col].imshow(cv2.cvtColor(distorted, cv2.COLOR_BGR2RGB))
    axes[0, col].set_title(f"{cam['nome']}\n{cam['desc']}", color='white', fontsize=10)
    axes[0, col].set_facecolor('#0f3460')
    axes[0, col].tick_params(colors='white')
    for sp in axes[0, col].spines.values():
        sp.set_edgecolor(cam['cor']); sp.set_linewidth(3)

    # Linha 1 — imagem corrigida (com calibração)
    new_K, _ = cv2.getOptimalNewCameraMatrix(
        K_ex, d, (SZ, SZ), 1, (SZ, SZ))
    corrected = cv2.undistort(distorted, K_ex, d, None, new_K)

    axes[1, col].imshow(cv2.cvtColor(corrected, cv2.COLOR_BGR2RGB))
    axes[1, col].set_title('Após calibração (corrigida)', color='#aaffaa', fontsize=10)
    axes[1, col].set_facecolor('#0f3460')
    axes[1, col].tick_params(colors='white')
    for sp in axes[1, col].spines.values():
        sp.set_edgecolor('#aaaaaa'); sp.set_linewidth(1)

axes[0, 0].set_ylabel('SEM calibração', color='#FF8A65', fontsize=12, fontweight='bold')
axes[1, 0].set_ylabel('COM calibração', color='#81C784', fontsize=12, fontweight='bold')

plt.suptitle(
    'Exercício: Comparando câmeras — distorção original vs. após calibração',
    color='white', fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

print('\n💬 QUESTÕES PARA REFLEXÃO:')
print('  1. Qual câmera você usaria SEM calibração? (Linha superior)')
print('  2. Após calibração, as 3 câmeras entregam resultados equivalentes?')
print('  3. A Câmera C (barata) vale a pena para inspeção industrial? Por quê?')
print('\n   → A resposta está nas imagens da linha inferior!')


---
## 1.5 Do Pixel ao Milímetro — O ROI da Calibração

Para tomadas de decisão em MBA, precisamos **quantificar o impacto**.

A célula abaixo responde: **"Quanto vale 1 pixel de erro de reprojeção no mundo físico?"**

Esta é a ponte entre a métrica técnica (RMS em pixels) e a decisão de negócio (aprovação ou rejeição de uma peça industrial).

> **Conceito-chave:** o erro RMS mede a precisão da *estimativa matemática dos parâmetros*. A partir do tamanho físico do tabuleiro e da distância focal estimada, podemos converter esse erro para milímetros — e compará-lo com tolerâncias industriais reais.


In [ ]:
# ─── 1.5 Do Pixel ao Milímetro: Impacto no Negócio ────────────────────────

# ── Parâmetros físicos do setup de calibração ────────────────────────────────
# Ajuste conforme o tabuleiro real utilizado
TAMANHO_QUADRADO_MM = 25.0          # cada quadrado do tabuleiro = 25 mm
N_QUADRADOS         = pattern_size[0]  # nº de cantos internos por linha

# Resolução física estimada: mm por pixel
# Derivação: fx (px) / N_QUADRADOS → px/quadrado → invertendo: mm/px
mm_por_px = TAMANHO_QUADRADO_MM / (fx / N_QUADRADOS)
erro_mm   = ret_rms * mm_por_px

# ── Benchmarks industriais (tolerâncias típicas) ─────────────────────────────
tolerancias = {
    'Inspeção automotiva (Tier 1)':   0.05,
    'Inspeção eletrônica (PCB)':      0.10,
    'Controle de qualidade geral':    0.25,
    'Medição dimensional básica':     0.50,
    'Vigilância / segurança':         2.00,
}

sep = '─' * 62
print('\n💰  ANÁLISE DE ROI — CALIBRAÇÃO DE CÂMERA')
print(sep)
print(f'  Tamanho do quadrado físico:  {TAMANHO_QUADRADO_MM:.1f} mm')
print(f'  Distância focal fx:          {fx:.1f} px')
print(f'  Resolução física estimada:   {mm_por_px:.4f} mm/pixel')
print(f'  Erro RMS de reprojeção:      {ret_rms:.4f} px')
print(f'  Erro físico correspondente:  {erro_mm:.4f} mm  ({erro_mm * 1000:.1f} µm)')
print()
print('  BENCHMARK — Requisitos de tolerância por aplicação:')
print(sep)

for aplicacao, tol_mm in tolerancias.items():
    status = '✅ APROVADO ' if erro_mm <= tol_mm else '❌ REPROVADO'
    print(f'  {status}  │  ±{tol_mm:.2f} mm  │  {aplicacao}')

print(sep)
print()

# ── Análise de impacto financeiro (simulação simplificada) ────────────────────
print('📊  IMPACTO FINANCEIRO (estimativa simplificada):')
print()
pecas_por_dia    = 10_000    # peças inspecionadas por dia
custo_peca_RS   = 50.0      # custo médio unitário
taxa_defeito     = 0.01      # 1 % de peças defeituosas

# Sem calibração: distorção não corrigida amplifica erro de medição
err_sem = min(abs(k1) * 0.5, 0.9)   # fração de peças mal avaliadas
fn_sem  = int(pecas_por_dia * taxa_defeito * err_sem)
custo_sem = fn_sem * custo_peca_RS * 5  # custo de recall ≈ 5× o custo da peça

# Com calibração
err_com = min(erro_mm / 10, 0.05)
fn_com  = int(pecas_por_dia * taxa_defeito * err_com)
custo_com = fn_com * custo_peca_RS * 5

economia = custo_sem - custo_com

print(f'  Cenário: {pecas_por_dia:,} peças/dia | Custo unitário: R${custo_peca_RS:.0f} | Defeitos: {taxa_defeito*100:.0f}%')
print()
print(f'  SEM calibração:')
print(f'    Falsos negativos estimados:   ~{fn_sem}/dia')
print(f'    Custo de recall:               R${custo_sem:,.0f}/dia')
print()
print(f'  COM calibração (RMS = {ret_rms:.3f} px = {erro_mm:.4f} mm):')
print(f'    Falsos negativos residuais:   ~{fn_com}/dia')
print(f'    Custo de recall:               R${custo_com:,.0f}/dia')
print()
print(f'  💡 ECONOMIA ESTIMADA:    R${economia:,.0f}/dia')
payback = 5000 / max(economia, 1)
print(f'  💡 Payback (câmera R$5.000): {payback:.1f} dias')
print(sep)


---
# Módulo 2 — Estimativa de Profundidade Monocular

## Contexto de Negócio

Saber **o quão longe** cada objeto está é fundamental para:
- **Robótica**: navegação autônoma sem colisões
- **E-commerce / AR**: visualização de produtos no ambiente real
- **Saúde**: suporte a cirurgias assistidas por robô

A abordagem clássica usa **duas câmeras** (stereo vision). O MiDaS resolve isso com **apenas uma câmera**, usando Deep Learning.

## MiDaS — Mixing Datasets for Zero-Shot Depth Estimation

| Aspecto | Detalhe |
|---------|---------|
| Criado por | Intel ISL |
| Arquitetura | Encoder-Decoder (DPT + ViT) |
| Treinamento | Múltiplos datasets misturados |
| Saída | **Mapa de profundidade relativo** |

```
Imagem RGB → Encoder (extrai features) → Decoder (reconstrói profundidade) → Mapa de profundidade
```

> A profundidade é **relativa** (não métrica): valores maiores = mais próximo ao observador.


In [ ]:
# Carregando MiDaS via PyTorch Hub
midas = torch.hub.load('intel-isl/MiDaS', 'MiDaS_small')
device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
midas.to(device)
midas.eval()

transforms_ = torch.hub.load('intel-isl/MiDaS', 'transforms')
transform = transforms_.small_transform

print(f" MiDaS carregado | Device: {device}")


In [ ]:
import requests

def baixar_imagem(url, caminho):
    response = requests.get(url)
    response.raise_for_status()
    with open(caminho, "wb") as f:
        f.write(response.content)
    print(f"Imagem salva: {caminho}")

def estimar_profundidade(imagem_bgr):
    """Aplica MiDaS e retorna mapa de profundidade normalizado [0,1]."""
    imgbatch = transform(imagem_bgr).to(device)
    with torch.no_grad():
        pred = midas(imgbatch)
        pred = torch.nn.functional.interpolate(
            pred.unsqueeze(1),
            size=imagem_bgr.shape[:2],
            mode="bicubic",
            align_corners=False,
        ).squeeze()
    output = pred.cpu().numpy()
    return (output - output.min()) / (output.max() - output.min())

# Baixar e processar imagem de exemplo
baixar_imagem(
    "https://howthingswork.org/wp-content/uploads/2017/05/Cars-down-the-road-e1494988061941.jpg",
    "imagem_estrada.jpg",
)

sample_img = cv2.imread("imagem_estrada.jpg")
depth_map  = estimar_profundidade(sample_img)

# Visualizacao lado a lado
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
axes[0].imshow(cv2.cvtColor(sample_img, cv2.COLOR_BGR2RGB))
axes[0].set_title("Imagem Original", fontsize=14)
axes[0].axis("off")

im = axes[1].imshow(depth_map, cmap="plasma")
axes[1].set_title("Mapa de Profundidade - MiDaS", fontsize=14)
axes[1].axis("off")
plt.colorbar(im, ax=axes[1], label="Profundidade relativa (claro = perto)")
plt.tight_layout()
plt.show()

### 2.1 Captura da Webcam (Colab Interativo)

O código abaixo integra JavaScript ao Colab para capturar imagens diretamente da webcam.


In [ ]:
VIDEO_HTML = """
<video autoplay width=800 height=600 style='cursor: pointer;'></video>
<script>
var video = document.querySelector('video');
navigator.mediaDevices.getUserMedia({ video: true })
  .then(stream => video.srcObject = stream);
var data = new Promise(resolve => {
  video.onclick = function() {
    var canvas = document.createElement('canvas');
    canvas.width = video.videoWidth; canvas.height = video.videoHeight;
    canvas.getContext('2d').drawImage(video, 0, 0);
    video.srcObject.getVideoTracks()[0].stop();
    video.replaceWith(canvas);
    resolve(canvas.toDataURL('image/jpeg', 0.8));
  };
});
</script>
"""

def take_photo(filename="photo.jpg"):
    """Captura foto pela webcam e retorna array numpy."""
    display(HTML(VIDEO_HTML))
    data = eval_js("data")
    img  = np.asarray(Image.open(io.BytesIO(b64decode(data.split(",")[1]))))
    cv2.imwrite(filename, cv2.cvtColor(img, cv2.COLOR_RGB2BGR))
    return img

# Para usar: clique no video para capturar
# img_webcam = take_photo()
# depth = estimar_profundidade(cv2.cvtColor(img_webcam, cv2.COLOR_RGB2BGR))
# plt.imshow(depth, cmap="plasma"); plt.colorbar(); plt.show()
print("Descomente as linhas acima para usar a webcam")

---
## Exercício — Segmentação por Camadas de Profundidade

**Objetivo:** Use o MiDaS para dividir a imagem `imagens/camara.jpg` em **5 fatias de profundidade** e visualize cada camada lado a lado.

**Dicas:**
1. Carregue a imagem com `cv2.imread()`
2. Converta BGR → RGB com `cv2.cvtColor(img, cv2.COLOR_BGR2RGB)`
3. Aplique `estimar_profundidade()` — já definida acima
4. Use `np.linspace(0, 1, num_slices + 1)` para criar os limiares
5. Para cada fatia: `mask = (depth >= limiar_inf) & (depth < limiar_sup)`

**Resultado esperado:** grade de imagens mostrando os diferentes planos da cena.


In [ ]:
import torch
import torchvision.transforms as T
import cv2
import matplotlib.pyplot as plt
import numpy as np

model_depth = torch.hub.load("intel-isl/MiDaS", "MiDaS_small")
model_depth.eval()

# COMPLETE O CODIGO ABAIXO

img     = # TODO: carregar imagem "imagens/camara.jpg"
img_rgb = # TODO: converter BGR para RGB
original_height, original_width, _ = # TODO: extrair dimensoes

transform_ex = T.Compose([
    T.ToPILImage(),
    T.Resize((384, 384)),
    T.ToTensor(),
    T.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5]),
])
input_batch = transform_ex(img_rgb).unsqueeze(0)

with torch.no_grad():
    prediction = model_depth(input_batch)

depth_map = prediction.squeeze().cpu().numpy()
depth_map = (depth_map - depth_map.min()) / (depth_map.max() - depth_map.min())
depth_map = cv2.resize(depth_map, (original_width, original_height))

num_slices   = # TODO: defina o numero de fatias (ex: 5)
depth_slices = np.linspace(0, 1, num_slices + 1)

plt.figure(figsize=(20, 8))
for i in range(num_slices):
    mask      = (depth_map >= depth_slices[i]) & (depth_map < depth_slices[i + 1])
    segmented = np.zeros_like(img_rgb)
    segmented[mask] = img_rgb[mask]

    plt.subplot(2, num_slices, i + 1)
    plt.imshow(segmented)
    plt.title(f"Camada {i + 1}", fontsize=10)
    plt.axis("off")

    depth_seg = np.zeros_like(depth_map)
    depth_seg[mask] = depth_map[mask]
    plt.subplot(2, num_slices, num_slices + i + 1)
    plt.imshow(depth_seg, cmap="plasma")
    plt.title(f"Mapa {i + 1}", fontsize=10)
    plt.axis("off")

plt.tight_layout()
plt.show()

---
# Módulo 3 — Deep Learning para Classificação de Imagens

## Dataset: COIL-100 (Columbia Object Image Library)

O COIL-100 contém **7.200 imagens** de **100 objetos diferentes**, cada um fotografado em 72 ângulos diferentes (0° a 358°).

![COIL-100](https://github.com/michelpf/fiap-ml-visao-computacional/blob/master/aula-5-machine-learning-aplicado/imagens/coil100.jpg?raw=true)

## Arquitetura CNN — Como Funciona?

```
Imagem (128×128×3)
 ↓
[Conv2D 32 filtros] → detecta bordas e texturas simples
 ↓
[MaxPooling 2×2] → reduz para 64×64, mantém features mais fortes
 ↓
[Conv2D 64 filtros] → detecta formas mais complexas
 ↓
[MaxPooling 2×2] → reduz para 32×32
 ↓
[Conv2D 128 filtros]→ features de alto nível (partes de objetos)
 ↓
[Flatten → Dense → Dropout → Softmax] → classificação final
```

### Intuição das camadas

| Camada | Analogia humana |
|--------|----------------|
| Conv2D (camada 1) | "Detectar bordas e contornos" |
| Conv2D (camada 2) | "Reconhecer texturas e padrões" |
| Conv2D (camada 3) | "Identificar partes do objeto" |
| Dense final | "Decidir qual objeto é" |


In [ ]:
import numpy as np, matplotlib.pyplot as plt, seaborn as sns, cv2
import datetime
from os import listdir
from os.path import join, isdir

import tensorflow as tf
from tensorflow.keras.models import Sequential, load_model
from tensorflow.keras.layers import Dense, Flatten, Conv2D, MaxPooling2D, Dropout
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input

print(f" TensorFlow {tf.__version__}")
print(f" GPU disponível: {len(tf.config.list_physical_devices('GPU')) > 0}")


In [ ]:
# Baixar dataset COIL-100
!rm -rf dataset-coil-100-adapted/
!git clone https://github.com/michelpf/dataset-coil-100-adapted --quiet
print(" Dataset COIL-100 baixado!")


### 3.1 Explorando o Dataset

Antes de modelar, sempre explore os dados: quantas classes? Há desequilíbrio? Qual a qualidade das imagens?


In [ ]:
pasta_treino = "dataset-coil-100-adapted/dataset"
classes = sorted([f for f in listdir(pasta_treino) if isdir(join(pasta_treino, f))])

print(f"Total de classes: {len(classes)}")
print(f"10 primeiras: {classes[:10]}")

# Amostras visuais
fig, axes = plt.subplots(2, 5, figsize=(14, 6))
for i, cls in enumerate(classes[:10]):
    imgs = [f for f in listdir(join(pasta_treino, cls)) if f.endswith(".png")]
    img  = cv2.imread(join(pasta_treino, cls, imgs[0]))
    axes[i // 5][i % 5].imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    axes[i // 5][i % 5].set_title(cls, fontsize=10)
    axes[i // 5][i % 5].axis("off")
plt.suptitle("Amostras do COIL-100 (10 primeiras classes)", fontsize=13)
plt.tight_layout()
plt.show()

### 3.2 Geradores de Dados

O `ImageDataGenerator` realiza **data augmentation** — criação de variações artificiais para enriquecer o treinamento.


In [ ]:
train_datagen = ImageDataGenerator(
    validation_split=0.3,
    preprocessing_function=preprocess_input,
)

train_generator = train_datagen.flow_from_directory(
    "dataset-coil-100-adapted/dataset",
    batch_size=32,
    class_mode="categorical",
    color_mode="rgb",
    target_size=(128, 128),
    subset="training",
)
validation_generator = train_datagen.flow_from_directory(
    "dataset-coil-100-adapted/dataset",
    batch_size=32,
    class_mode="categorical",
    color_mode="rgb",
    target_size=(128, 128),
    subset="validation",
)

print(f"\nAmostras de treino:    {train_generator.samples}")
print(f"Amostras de validacao: {validation_generator.samples}")
print(f"Classes: {len(train_generator.class_indices)}")

### 3.3 Arquitetura CNN

Cada camada tem um papel na extração progressiva de features:

- **ReLU**: ativa apenas neurônios com resposta positiva → introduz não-linearidade
- **MaxPooling**: comprime a feature map mantendo o valor máximo local → reduz ruído
- **Dropout 0.5**: desliga 50% dos neurônios aleatoriamente durante treino → evita overfitting
- **Softmax**: converte o vetor final em probabilidades (soma = 1)


In [ ]:
n_classes = len(classes)

model = Sequential([
    # Bloco 1 - bordas e padroes simples
    Conv2D(32, (3, 3), activation="relu", padding="same", input_shape=(128, 128, 3)),
    MaxPooling2D(2, 2),

    # Bloco 2 - formas mais complexas
    Conv2D(64, (3, 3), activation="relu", padding="same"),
    MaxPooling2D(2, 2),

    # Bloco 3 - features de alto nivel
    Conv2D(128, (3, 3), activation="relu", padding="same"),
    MaxPooling2D(2, 2),

    # Classificador
    Flatten(),
    Dense(256, activation="relu"),
    Dropout(0.5),
    Dense(n_classes, activation="softmax"),
])

model.summary()
print(f"\nTotal de parametros treinaveis: {model.count_params():,}")

### 3.4 Visualizando a Arquitetura e Filtros


In [ ]:
!pip install visualkeras==0.1.4 --quiet
import visualkeras
visualkeras.layered_view(model, legend=True, to_file="arquitetura_cnn.png")
print(" Diagrama da arquitetura salvo!")


In [ ]:
# Filtros da 1a camada (padroes que o modelo aprendera a detectar)
filters, _ = model.layers[0].get_weights()
filters_norm = (filters - filters.min()) / (filters.max() - filters.min())

fig, axes = plt.subplots(4, 8, figsize=(16, 8))
for idx, ax in enumerate(axes.flat):
    if idx < filters_norm.shape[-1]:
        ax.imshow(filters_norm[:, :, :, idx])
    ax.axis("off")

plt.suptitle(
    "Filtros da 1a camada convolucional (antes do treinamento)\n"
    "Apos o treino, cada filtro aprendera a detectar um padrao especifico",
    fontsize=12,
)
plt.tight_layout()
plt.show()

### 3.5 Treinamento e Curvas de Aprendizado

> **O que monitorar?**
> - **Acurácia de treino e validação** devem convergir
> - Se treino >> validação: **overfitting** → aumente dropout ou reduza modelo
> - Se ambas baixas: **underfitting** → aumente épocas ou capacidade do modelo


In [ ]:
model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

history = model.fit(
    train_generator,
    validation_data=validation_generator,
    epochs=3,
    verbose=1
)


In [ ]:
# Curvas de aprendizado
sns.set_style("darkgrid")
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(history.history['accuracy'], marker='o', label='Treino')
axes[0].plot(history.history['val_accuracy'], marker='o', label='Validação')
axes[0].set_title('Acurácia por Época', fontsize=13)
axes[0].set_xlabel('Época'); axes[0].set_ylabel('Acurácia')
axes[0].legend()

axes[1].plot(history.history['loss'], marker='o', label='Treino')
axes[1].plot(history.history['val_loss'], marker='o', label='Validação')
axes[1].set_title('Loss (Erro) por Época', fontsize=13)
axes[1].set_xlabel('Época'); axes[1].set_ylabel('Loss')
axes[1].legend()

plt.tight_layout()
plt.show()


In [ ]:
import os
os.makedirs("modelos", exist_ok=True)
os.makedirs("pesos", exist_ok=True)

model.save("modelos/model_coil100.h5")
model.save_weights("pesos/pesos_coil100.weights.h5")
print(" Modelo salvo! (Para recarregar: model = load_model('modelos/model_coil100.h5'))")


### 3.6 Inferência e Resultados


In [ ]:
imagens_teste = [
    "dataset-coil-100-adapted/dataset/alien-toy/20.png",
    "dataset-coil-100-adapted/dataset/telephone/20.png",
    "dataset-coil-100-adapted/dataset/cheeseburger/20.png",
]

lista_imgs = []
lista_arrs = []
for path in imagens_teste:
    img = cv2.cvtColor(cv2.imread(path), cv2.COLOR_BGR2RGB)
    lista_imgs.append(img)
    lista_arrs.append(cv2.resize(img, (128, 128)))

arr        = preprocess_input(np.array(lista_arrs, dtype="float32"))
pred_probs = model.predict(arr)
y_classes  = pred_probs.argmax(axis=-1)

key_list = list(train_generator.class_indices.keys())
val_list = list(train_generator.class_indices.values())

fig, axes = plt.subplots(1, len(lista_imgs), figsize=(14, 5))
for i, (img, pred_idx) in enumerate(zip(lista_imgs, y_classes)):
    nome      = key_list[val_list.index(pred_idx)]
    confianca = pred_probs[i].max() * 100
    axes[i].imshow(img)
    axes[i].set_title(f"Predito: {nome}\n{confianca:.1f}% confianca", fontsize=11)
    axes[i].axis("off")
plt.suptitle("Classificacao CNN - COIL-100", fontsize=14)
plt.tight_layout()
plt.show()

### 3.7 Explicabilidade com GradCAM

**GradCAM** (Gradient-weighted Class Activation Mapping) responde à pergunta:
*"Qual região da imagem foi mais determinante para a classificação?"*

É fundamental para **validar se o modelo aprendeu pelos motivos corretos** — e não está usando atalhos como o fundo da imagem.

 [Paper original: Selvaraju et al., 2017](https://arxiv.org/abs/1610.02391)

> *Exemplo clássico do problema*: um modelo aprende a classificar "girafa" pelo fundo árido da savana africana — não pelo animal em si. O GradCAM revelaria isso antes do deploy.


---
# Módulo 4 — Transfer Learning para Reconhecimento de Imagens

## Por que Transfer Learning?

Treinar uma rede do zero exige:
- Milhões de imagens rotuladas
- Semanas de treinamento em clusters de GPU
- Alto custo financeiro e ambiental

O **Transfer Learning** reutiliza features aprendidas em datasets gigantes e adapta apenas as últimas camadas.

```
ImageNet (1.2M imagens, 1000 classes)
 ↓
[VGG19 — aprende hierarquia de features]
 camada 1: bordas simples
 camada 5: texturas
 camada 10: partes de objetos
 camada 19: objetos completos
 ↓
[CONGELAR — layer.trainable = False]
 ↓
[Nova camada Dense — treinada do zero para sua tarefa]
```

## Dataset: American Sign Language Digits

Dígitos 0–9 em linguagem de sinais americana.
**5.000 imagens | 10 classes | 400×400 pixels**


In [ ]:
from tensorflow.keras.applications import VGG19
from tensorflow.keras import layers, Model
from tensorflow.keras.models import model_from_json

!rm -rf dataset-american-sign-language-digit/
!git clone https://github.com/michelpf/dataset-american-sign-language-digit --quiet
print(" Dataset ASL baixado!")


In [ ]:
# Visualizar amostras do dataset
fig, axes = plt.subplots(2, 5, figsize=(14, 6))
for digito in range(10):
    path = f"dataset-american-sign-language-digit/dataset/{digito}/Sign {digito} (1).jpeg"
    img  = cv2.cvtColor(cv2.imread(path), cv2.COLOR_BGR2RGB)
    ax   = axes[digito // 5][digito % 5]
    ax.imshow(img)
    ax.set_title(f"Digito: {digito}", fontsize=12)
    ax.axis("off")
plt.suptitle("Dataset - American Sign Language Digits", fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Geradores com Data Augmentation
train_datagen_tl = ImageDataGenerator(
    shear_range=10,
    zoom_range=0.2,
    horizontal_flip=True,
    validation_split=0.3,
    preprocessing_function=preprocess_input,
)

train_gen_tl = train_datagen_tl.flow_from_directory(
    "dataset-american-sign-language-digit/dataset",
    batch_size=32,
    class_mode="categorical",
    color_mode="rgb",
    target_size=(128, 128),
    subset="training",
)
val_gen_tl = train_datagen_tl.flow_from_directory(
    "dataset-american-sign-language-digit/dataset",
    batch_size=32,
    class_mode="categorical",
    color_mode="rgb",
    target_size=(128, 128),
    subset="validation",
)
print(f"Classes: {train_gen_tl.class_indices}")

### 4.1 Arquitetura com VGG19

O parâmetro `include_top=False` remove as camadas densas do VGG19 — preservamos apenas o **extrator de features** (backbone).


In [ ]:
# VGG19 sem as camadas densas - so o extrator de features
conv_base = VGG19(include_top=False, input_shape=(128, 128, 3))

# Congelar TODOS os pesos do VGG19
for layer in conv_base.layers:
    layer.trainable = False

print(f"Camadas do VGG19: {len(conv_base.layers)} (todas congeladas)")

# Adicionar camadas de classificacao no topo
x           = conv_base.output
x           = layers.GlobalAveragePooling2D()(x)
x           = layers.Dense(128, activation="relu")(x)
predictions = layers.Dense(10, activation="softmax")(x)

model_tl = Model(conv_base.input, predictions)
model_tl.summary()

In [ ]:
model_tl.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
history_tl = model_tl.fit(train_gen_tl, epochs=5, validation_data=val_gen_tl, verbose=1)


In [ ]:
# Curvas de aprendizado — Transfer Learning
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(history_tl.history['accuracy'], marker='o', label='Treino')
axes[0].plot(history_tl.history['val_accuracy'], marker='o', label='Validação')
axes[0].set_title('Acurácia — VGG19 Fine-tuning', fontsize=13)
axes[0].legend()

axes[1].plot(history_tl.history['loss'], marker='o', label='Treino')
axes[1].plot(history_tl.history['val_loss'], marker='o', label='Validação')
axes[1].set_title('Loss — VGG19 Fine-tuning', fontsize=13)
axes[1].legend()
plt.tight_layout()
plt.show()


In [ ]:
model_tl.save('modelos/modelo-sign.h5')
print(" Modelo de linguagem de sinais salvo!")


### 4.2 Predições e Avaliação


In [ ]:
lista_imgs_asl = []
lista_arrs_asl = []
for i in range(10):
    path = f"dataset-american-sign-language-digit/dataset/{i}/Sign {i} (1).jpeg"
    img  = cv2.cvtColor(cv2.imread(path), cv2.COLOR_BGR2RGB)
    lista_imgs_asl.append(img)
    lista_arrs_asl.append(cv2.resize(img, (128, 128)))

arr_asl    = preprocess_input(np.array(lista_arrs_asl, dtype="float32"))
pred_asl   = model_tl.predict(arr_asl)
y_pred_asl = pred_asl.argmax(axis=-1)

fig, axes = plt.subplots(2, 5, figsize=(16, 8))
for i, (img, pred) in enumerate(zip(lista_imgs_asl, y_pred_asl)):
    ax   = axes[i // 5][i % 5]
    conf = pred_asl[i].max() * 100
    acerto = "Certo" if pred == i else "Errado"
    ax.imshow(img)
    ax.set_title(f"[{acerto}] Real:{i} | Pred:{pred}\n{conf:.1f}%", fontsize=10)
    ax.axis("off")
plt.suptitle("Transfer Learning - ASL Digit Recognition (VGG19)", fontsize=14)
plt.tight_layout()
plt.show()

---
# Módulo 5 — Detecção de Objetos com YOLO

## Classificação vs. Detecção vs. Segmentação

| Tarefa | Pergunta respondida | Saída |
|--------|---------------------|-------|
| **Classificação** | "O que está na imagem?" | Classe + confiança |
| **Detecção** | "O que está e ONDE está?" | Bounding boxes + classes |
| **Segmentação** | "Qual pixel pertence a quê?" | Máscara por pixel |

## YOLO — You Only Look Once

O YOLO revolucionou a detecção de objetos ao processar **a imagem inteira em uma única passagem** pela rede neural.

```
Arquiteturas anteriores: YOLO:
1. Propor regiões candidatas → Processar imagem completa de uma vez
2. Classificar cada região Prever boxes + classes simultaneamente
Resultado: ~0.5 FPS Resultado: 30–100+ FPS em tempo real
```

### Evolução do YOLO

| Versão | Ano | Novidade |
|--------|-----|---------|
| YOLOv1 | 2016 | Conceito "you only look once" |
| YOLOv5 | 2020 | Alta adoção, fácil de usar |
| YOLOv8 | 2023 | Suporte a seg. + pose |
| **YOLO12** | **2025** | **Atenção + backbone otimizado** |


In [ ]:
!pip install ultralytics --quiet
import ultralytics
from ultralytics import YOLO

# Verificar instalação e detecção de GPU
ultralytics.checks()


### 5.1 Detecção com Modelo Pré-treinado (COCO — 80 classes)


In [ ]:
# Carregar YOLO12 nano (2.6M parametros - rapido e leve)
model_yolo = YOLO("yolo12n.pt")

# Inferencia em imagem de cachorro
results = model_yolo.predict(source="imagens/dog2.jpeg", conf=0.5, verbose=False)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
img_orig = cv2.imread("imagens/dog2.jpeg")
axes[0].imshow(cv2.cvtColor(img_orig, cv2.COLOR_BGR2RGB))
axes[0].set_title("Imagem Original", fontsize=13)
axes[0].axis("off")

axes[1].imshow(cv2.cvtColor(results[0].plot(), cv2.COLOR_BGR2RGB))
n_objs = len(results[0].boxes)
axes[1].set_title(f"YOLO12: {n_objs} objeto(s) detectado(s)", fontsize=13)
axes[1].axis("off")
plt.tight_layout()
plt.show()

# Detalhes das deteccoes
print("\nDetalhes:")
for box in results[0].boxes:
    cls  = results[0].names[int(box.cls)]
    conf = float(box.conf)
    print(f"  {cls}: {conf:.1%}")

In [ ]:
# Detecção em cena de escritório — múltiplos objetos
img_office = cv2.imread("imagens/office.jpg")
res_office = model_yolo.predict(source=img_office, conf=0.4, verbose=False)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
axes[0].imshow(cv2.cvtColor(img_office, cv2.COLOR_BGR2RGB))
axes[0].set_title("Cena Original", fontsize=13); axes[0].axis('off')
axes[1].imshow(cv2.cvtColor(res_office[0].plot(), cv2.COLOR_BGR2RGB))
axes[1].set_title(f"Detecção: {len(res_office[0].boxes)} objetos", fontsize=13)
axes[1].axis('off')
plt.tight_layout()
plt.show()


### 5.2 Treinamento de Modelo Customizado

**Caso de uso real:** Uma prefeitura quer detectar **buracos nas ruas** automaticamente usando câmeras em ônibus municipais para priorizar manutenção.

Dataset: [Potholes Detection YOLO](https://www.kaggle.com/datasets/anugrahakbar/potholes-detection-for-yolov4)

#### Estrutura de um dataset YOLO:
```
dataset/
 train/
 images/ ← imagens .jpg
 labels/ ← arquivos .txt: "classe x_center y_center largura altura" (normalizado 0-1)
 test/
 images/
 labels/
```


In [ ]:
!git clone https://github.com/michelpf/dataset-pothole --quiet
print(" Dataset de buracos baixado!")


In [ ]:
%%writefile configs_modelo.yaml
path: '/content/fiap-ml-visao-computacional/aula-5-machine-learning-aplicado/dataset-pothole/dataset'
train: 'train/'
val: 'test/'
nc: 1
names: ["pothole"]


In [ ]:
# Treinar YOLO12 customizado - 10 epocas para demonstracao
# Em producao: 50-100 epocas para melhor performance
resultados = model_yolo.train(
    data="configs_modelo.yaml",
    epochs=10,
    imgsz=640,
    name="yolov12_pothole",
    verbose=False,
)
print("Treinamento concluido!")

### 5.3 Análise das Métricas de Treinamento

O YOLO gera automaticamente curvas de desempenho:

| Gráfico | O que avalia |
|---------|-------------|
| **Precision** | Das detecções feitas, quantas eram corretas? |
| **Recall** | Dos objetos reais, quantos foram detectados? |
| **F1-Score** | Balanço entre Precision e Recall |
| **mAP@0.5** | Área sob a curva Precision-Recall (métrica padrão COCO) |


In [ ]:
run_dir = "/content/fiap-ml-visao-computacional/aula-5-machine-learning-aplicado/runs/detect/yolov12_pothole"

graficos = {
    "BoxF1_curve.png": "Curva F1",
    "BoxPR_curve.png": "Curva PR",
    "BoxP_curve.png":  "Precision",
    "BoxR_curve.png":  "Recall",
    "results.png":     "Resultados",
}

fig, axes = plt.subplots(1, len(graficos), figsize=(20, 4))
for ax, (fname, titulo) in zip(axes, graficos.items()):
    img = cv2.imread(f"{run_dir}/{fname}")
    if img is not None:
        ax.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
        ax.set_title(titulo, fontsize=11)
    ax.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
# Inferencia com modelo customizado em imagens novas
dir_pesos    = f"{run_dir}/weights/best.pt"
model_custom = YOLO(dir_pesos)

imgs_buraco = ["imagens/buraco.jpg", "imagens/buraco2.jpeg"]
fig, axes   = plt.subplots(1, len(imgs_buraco) * 2, figsize=(16, 5))

for i, path in enumerate(imgs_buraco):
    img = cv2.cvtColor(cv2.imread(path), cv2.COLOR_BGR2RGB)
    axes[i * 2].imshow(img)
    axes[i * 2].set_title("Original", fontsize=11)
    axes[i * 2].axis("off")

    res = model_custom.predict(source=img, conf=0.25, verbose=False)
    axes[i * 2 + 1].imshow(cv2.cvtColor(res[0].plot(), cv2.COLOR_BGR2RGB))
    axes[i * 2 + 1].set_title(f"{len(res[0].boxes)} buraco(s)", fontsize=11)
    axes[i * 2 + 1].axis("off")

plt.suptitle("Deteccao de Buracos - Modelo Customizado YOLO12", fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Funcoes de inferencia para producao

def predict_single_image(image_path, model, conf=0.25):
    """Inferencia em imagem unica com visualizacao."""
    results = model(image_path, conf=conf, verbose=False)
    img     = cv2.cvtColor(cv2.imread(image_path), cv2.COLOR_BGR2RGB)
    res     = cv2.cvtColor(results[0].plot(), cv2.COLOR_BGR2RGB)
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
    ax1.imshow(img)
    ax1.set_title("Original")
    ax1.axis("off")
    ax2.imshow(res)
    ax2.set_title("Deteccao")
    ax2.axis("off")
    plt.tight_layout()
    plt.show()
    return results


def predict_batch(image_folder, model, batch_size=16, conf=0.25):
    """Inferencia em lote: processa todas as imagens de uma pasta."""
    paths = [
        os.path.join(image_folder, f)
        for f in os.listdir(image_folder)
        if f.lower().endswith((".png", ".jpg", ".jpeg"))
    ]
    print(f"{len(paths)} imagens encontradas")
    for i in range(0, len(paths), batch_size):
        model(paths[i : i + batch_size], conf=conf, verbose=False)
        print(f"  Lote {i // batch_size + 1} processado")


def process_video(video_path, model, output_path=None, conf=0.25):
    """Aplica deteccao frame a frame em um video."""
    if not output_path:
        output_path = os.path.splitext(video_path)[0] + "_detected.mp4"
    cap = cv2.VideoCapture(video_path)
    w   = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    h   = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    out = cv2.VideoWriter(
        output_path,
        cv2.VideoWriter_fourcc(*"mp4v"),
        cap.get(cv2.CAP_PROP_FPS),
        (w, h),
    )
    n = 0
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break
        out.write(model(frame, verbose=False)[0].plot())
        n += 1
    cap.release()
    out.release()
    print(f"{n} frames processados -> {output_path}")


print("Funcoes de producao definidas!")

In [ ]:
# Exportar para ONNX — deploy em produção (C++, mobile, TensorRT)
model_custom.export(format='onnx')
print(" Exportado para ONNX!")
print(" Compatível com: TensorRT, OpenCV DNN, ONNX Runtime, C++, iOS/Android")


---
# Módulo 6 — Segmentação de Imagens

## Detecção vs. Segmentação

| | Detecção (YOLO) | Segmentação |
|-|----------------|-------------|
| **Precisão espacial** | Retângulo (bounding box) | Contorno exato pixel a pixel |
| **Casos de uso** | Tempo real, IoT, rastreamento | Medicina, análise detalhada |
| **Custo computacional** | Baixo | Alto |

Neste módulo: **RF-DETR** (Transformers) e **SAM** (Meta AI — segmenta qualquer coisa).


## 6.1 RF-DETR — Detection Transformer da Roboflow

O RF-DETR foi o primeiro modelo a superar **60 AP no benchmark COCO** com latência competitiva.

| Característica | Detalhe |
|---------------|---------|
| Base | DETR (DEtection TRansformer) |
| Destaque | 60+ AP no COCO, excelente em domínios customizados |
| Vantagem | Zero-shot para novos domínios sem fine-tuning |


In [ ]:
!pip install rfdetr supervision --quiet

import supervision as sv
from rfdetr import RFDETRSegMedium
from rfdetr.assets.coco_classes import COCO_CLASSES

model_rfdetr = RFDETRSegMedium()

image_path = "imagens/pessoas2.jpg"
image_bgr = cv2.imread(image_path)
image_rgb = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)

detections = model_rfdetr.predict(image_path, threshold=0.5)
labels = [f"{COCO_CLASSES[cid]}" for cid in detections.class_id]

annotated = sv.MaskAnnotator().annotate(scene=image_rgb.copy(), detections=detections)
annotated = sv.LabelAnnotator().annotate(scene=annotated, detections=detections, labels=labels)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
axes[0].imshow(image_rgb); axes[0].set_title("Original", fontsize=13); axes[0].axis('off')
axes[1].imshow(annotated); axes[1].set_title("RF-DETR Segmentation", fontsize=13); axes[1].axis('off')
plt.tight_layout()
plt.show()


## 6.2 SAM — Segment Anything Model (Meta AI, 2023)

O SAM é um **modelo fundacional** de segmentação:
- Treinado em **1 bilhão de máscaras**
- Capaz de segmentar **qualquer objeto** sem retreinamento
- Suporta múltiplos modos de prompt

### Modos de Operação SAM

| Modo | Como funciona | Quando usar |
|------|--------------|-------------|
| **Automático** | Segmenta tudo na imagem | Exploração, anotação de dados |
| **Por ponto** | Clique em um pixel | Interface do usuário |
| **Por bounding box** | Define ROI | Combinado com YOLO |


In [ ]:
!pip install -q 'git+https://github.com/facebookresearch/segment-anything.git'
print(" SAM instalado!")


In [ ]:
import os
HOME = os.getcwd()

# Baixar pesos do SAM ViT-H (modelo mais preciso, ~2.4 GB)
!mkdir -p {HOME}/weights
!wget -q https://dl.fbaipublicfiles.com/segment_anything/sam_vit_h_4b8939.pth -P {HOME}/weights

!mkdir -p {HOME}/data
!wget -q https://media.roboflow.com/notebooks/examples/dog.jpeg -P {HOME}/data
!wget -q https://media.roboflow.com/notebooks/examples/dog-2.jpeg -P {HOME}/data

print(" Pesos e imagens de teste baixados!")


In [ ]:
import torch
from segment_anything import sam_model_registry, SamAutomaticMaskGenerator, SamPredictor

DEVICE     = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
MODEL_TYPE = "vit_h"

sam = sam_model_registry[MODEL_TYPE](
    checkpoint=f"{HOME}/weights/sam_vit_h_4b8939.pth"
).to(device=DEVICE)

print(f"SAM carregado | Device: {DEVICE}")

### 6.2.1 Segmentação Automática

`SamAutomaticMaskGenerator` analisa a imagem toda e gera máscaras para **cada objeto detectado** automaticamente.


In [ ]:
mask_generator = SamAutomaticMaskGenerator(sam)

IMAGE_PATH = os.path.join(HOME, "data", "dog.jpeg")
image_bgr = cv2.imread(IMAGE_PATH)
image_rgb = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)

sam_result = mask_generator.generate(image_rgb)
print(f" Total de regiões segmentadas: {len(sam_result)}")
print(f" Campos de cada máscara: {list(sam_result[0].keys())}")


In [ ]:
mask_annotator = sv.MaskAnnotator(color_lookup=sv.ColorLookup.INDEX)
detections_sam = sv.Detections.from_sam(sam_result=sam_result)
annotated_sam = mask_annotator.annotate(scene=image_bgr.copy(), detections=detections_sam)

sv.plot_images_grid(
    images=[image_bgr, annotated_sam],
    grid_size=(1, 2),
    titles=['Imagem Original', f'SAM: {len(sam_result)} máscaras'],
    size=(16, 8)
)


In [ ]:
# Visualizar máscaras individuais (ordenadas por área)
masks_sorted = [m['segmentation'] for m in sorted(sam_result, key=lambda x: x['area'], reverse=True)]
n_show = min(32, len(masks_sorted))

sv.plot_images_grid(
    images=masks_sorted[:n_show],
    grid_size=(4, 8),
    size=(16, 8)
)


### 6.2.2 Segmentação por Bounding Box (Interativa)

O `SamPredictor` usa uma bounding box como **prompt** para segmentar um objeto específico.

> **Pipeline poderoso**: YOLO detecta os objetos → SAM segmenta com precisão de pixel.


In [ ]:
import base64
import numpy as np

def encode_image(filepath):
    with open(filepath, "rb") as f:
        return "data:image/jpg;base64," + base64.b64encode(f.read()).decode("utf-8")

mask_predictor = SamPredictor(sam)

!pip install jupyter_bbox_widget --quiet

In [ ]:
IS_COLAB   = True
IMAGE_PATH = os.path.join(HOME, "data", "dog.jpeg")

if IS_COLAB:
    from google.colab import output
    output.enable_custom_widget_manager()

from jupyter_bbox_widget import BBoxWidget
widget = BBoxWidget()
widget.image = encode_image(IMAGE_PATH)
widget
# Desenhe um retangulo ao redor do cachorro e execute a proxima celula

In [ ]:
# Usar bounding box desenhada (ou o default abaixo)
default_box = {"x": 68, "y": 247, "width": 555, "height": 678, "label": ""}
box = widget.bboxes[0] if widget.bboxes else default_box
box = np.array([
    box["x"],
    box["y"],
    box["x"] + box["width"],
    box["y"] + box["height"],
])
print(f"Bounding box: {box}")

In [ ]:
# Segmentacao guiada pela bounding box
image_bgr = cv2.imread(IMAGE_PATH)
image_rgb = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)
mask_predictor.set_image(image_rgb)

masks, scores, logits = mask_predictor.predict(
    box=box,
    multimask_output=True,  # retorna 3 mascaras candidatas
)
print(f"{len(masks)} mascaras geradas | Scores: {[f'{s:.3f}' for s in scores]}")

In [ ]:
box_annotator = sv.BoxAnnotator(color=sv.Color.RED, color_lookup=sv.ColorLookup.INDEX)
mask_annotator = sv.MaskAnnotator(color=sv.Color.RED, color_lookup=sv.ColorLookup.INDEX)

detections = sv.Detections(xyxy=sv.mask_to_xyxy(masks=masks), mask=masks)
detections = detections[detections.area == np.max(detections.area)]

source_img = box_annotator.annotate(scene=image_bgr.copy(), detections=detections)
segmented_img = mask_annotator.annotate(scene=image_bgr.copy(), detections=detections)

sv.plot_images_grid(
    images=[source_img, segmented_img],
    grid_size=(1, 2),
    titles=['Bounding Box (input)', 'Segmentação SAM (output)'],
    size=(14, 7)
)


---
# Módulo 7 — Vision Transformer (ViT)

## Da CNN ao Transformer

As CNNs têm **campo receptivo local** — enxergam regiões pequenas por vez e combinam informações gradualmente.

O **Vision Transformer** aplica **atenção** diretamente em patches da imagem:

```
Imagem 224×224
 ↓
Dividir em patches 16×16 → 196 patches
 ↓
Cada patch = "token" (como palavras no BERT/GPT)
 ↓
Multi-Head Self-Attention → cada patch "olha" para todos os outros
 ↓
Token [CLS] → classificação final
```

## CNN vs. ViT

| Aspecto | CNN | ViT |
|---------|-----|-----|
| Dados necessários | Menos (bom com poucos dados) | Mais (ou fine-tuning) |
| Relações globais | Limitado (local → global gradual) | Excelente (global desde a 1ª camada) |
| Velocidade (inferência) | Rápida | Mais lenta |
| Explicabilidade | GradCAM | **Attention Rollout** |
| Escalabilidade | Boa | Excelente (quanto mais dados, melhor) |

## Dataset: Diagnóstico de Tuberculose por Raio-X

Problema binário: **Normal vs. Tuberculose** — exatamente onde ViT brilha, pois relações globais na imagem são essenciais para diagnóstico.


In [ ]:
!pip install linformer vit_pytorch timm --quiet
print(" Dependências do ViT instaladas!")


In [ ]:
import copy, math
import matplotlib.pyplot as plt
import numpy as np, pandas as pd, seaborn as sns
import timm, torch, torch.nn as nn, torch.optim as optim
from linformer import Linformer
from PIL import Image as PILImage
from sklearn import metrics
from sklearn.metrics import confusion_matrix
from torch.optim.lr_scheduler import StepLR
from torchvision import datasets, transforms
from tqdm import tqdm
from vit_pytorch import ViT

print(" Importações do Módulo 7 concluídas!")


### 7.1 Classe VisionClassifier — Pipeline Completo

Esta classe encapsula todo o pipeline de ViT:
- Carregamento e divisão treino/validação
- Suporte a ViT padrão (timm) e ViT com Linformer
- Treinamento com scheduler de learning rate
- Avaliação: Acurácia, F1, MCC, AUC-ROC + Curva ROC + Matriz de Confusão
- **Attention Rollout** para interpretabilidade


In [ ]:
# VisionClassifier - Vision Transformer + Attention Rollout


class VisionClassifier:
    def __init__(
        self,
        data_dir,
        batch_size=32,
        val_split=0.3,
        num_workers=4,
        image_size=224,
        seed=0,
        use_linformer=False,
    ):
        """
        Classificador de imagens baseado em Vision Transformer (ViT).

        Args:
            data_dir (str): Diretorio com imagens organizadas em pastas por classe
            batch_size (int): Tamanho do batch
            val_split (float): Proporcao para validacao (0-1)
            image_size (int): Tamanho das imagens (224 = padrao ViT)
            seed (int): Para reprodutibilidade
            use_linformer (bool): True = ViT com Linformer (eficiente)
        """
        self.data_dir      = data_dir
        self.batch_size    = batch_size
        self.val_split     = val_split
        self.num_workers   = num_workers
        self.image_size    = image_size
        self.seed          = seed
        self.use_linformer = use_linformer
        self.device        = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

        self.train_loss_history = []
        self.train_acc_history  = []
        self.val_loss_history   = []
        self.val_acc_history    = []
        self.best_acc           = 0.0

        self._setup_data()
        self._setup_model()

    # ── Setup de dados ────────────────────────────────────────────────────

    def _setup_data(self):
        self.dataset_completo = datasets.ImageFolder(self.data_dir)
        self.class_names      = self.dataset_completo.classes
        print(f"Classes: {self.class_names}")

        n  = len(self.dataset_completo)
        nt = int((1 - self.val_split) * n)
        torch.manual_seed(self.seed)
        self.train_dataset, self.val_dataset = torch.utils.data.random_split(
            self.dataset_completo, [nt, n - nt]
        )
        self.dataset_sizes = {"train": nt, "val": n - nt}
        print(f"Tamanhos: {self.dataset_sizes}")

        tf = transforms.Compose([
            transforms.Resize((self.image_size, self.image_size)),
            transforms.ToTensor(),
            transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
        ])
        self.data_transforms = {"train": tf, "val": tf}
        self.train_dataset.dataset.transform = tf

        self.train_dataloader = torch.utils.data.DataLoader(
            self.train_dataset,
            batch_size=self.batch_size,
            shuffle=True,
            num_workers=self.num_workers,
        )
        self.val_dataloader = torch.utils.data.DataLoader(
            self.val_dataset,
            batch_size=self.batch_size,
            shuffle=True,
            num_workers=self.num_workers,
        )
        self.dataloaders = {"train": self.train_dataloader, "val": self.val_dataloader}

    # ── Setup do modelo ───────────────────────────────────────────────────

    def _setup_model(self):
        if self.use_linformer:
            d  = 128
            et = Linformer(dim=d, seq_len=49 + 1, depth=12, heads=8, k=64)
            self.modelo = ViT(
                dim=d,
                image_size=self.image_size,
                patch_size=32,
                num_classes=2,
                transformer=et,
                channels=3,
            )
            num_ftrs = d
        else:
            self.modelo = timm.create_model("vit_base_patch16_224", pretrained=True)
            num_ftrs    = self.modelo.head.in_features

        self.modelo.head = nn.Linear(num_ftrs, len(self.class_names))
        self.modelo      = self.modelo.to(self.device)

        params = sum(p.numel() for p in self.modelo.parameters() if p.requires_grad)
        print(f"Parametros treinaveis: {params:,}")

        self.criterion = nn.CrossEntropyLoss()
        self.optimizer = optim.Adam(self.modelo.parameters(), lr=3e-5)
        self.scheduler = StepLR(self.optimizer, step_size=7, gamma=0.1)

    # ── Visualizacao de batch ─────────────────────────────────────────────

    def visualize_batch(self, num_rows=4, num_cols=8):
        inputs, labels = next(iter(self.train_dataloader))
        inputs         = inputs.numpy()
        labels         = labels.numpy()
        label_names    = [self.class_names[l] for l in labels]

        fig, axes = plt.subplots(figsize=(14, 8), nrows=num_rows, ncols=num_cols)
        for i, ax in enumerate(axes.flat):
            if i >= len(inputs):
                break
            img = np.clip(
                np.transpose(inputs[i], (1, 2, 0)) * [0.229, 0.224, 0.225]
                + [0.485, 0.456, 0.406],
                0,
                1,
            )
            ax.imshow(img)
            ax.axis("off")
            ax.set_title(label_names[i], fontsize=8)
        plt.tight_layout()
        plt.show()

    # ── Treinamento ───────────────────────────────────────────────────────

    def train(self, num_epochs=15):
        best_wts = copy.deepcopy(self.modelo.state_dict())

        for epoch in range(num_epochs):
            print(f"Epoch {epoch}/{num_epochs - 1}")
            print("-" * 10)

            for phase in ["train", "val"]:
                if phase == "train":
                    self.modelo.train()
                else:
                    self.modelo.eval()

                running_loss      = 0.0
                running_corrects  = 0

                with tqdm(self.dataloaders[phase], unit="batch") as t:
                    for inputs, labels in t:
                        t.set_description(f"Epoch {epoch} {phase}")
                        inputs = inputs.to(self.device)
                        labels = labels.to(self.device)
                        self.optimizer.zero_grad()

                        with torch.set_grad_enabled(phase == "train"):
                            out       = self.modelo(inputs)
                            _, preds  = torch.max(out, 1)
                            loss      = self.criterion(out, labels)
                            if phase == "train":
                                loss.backward()
                                self.optimizer.step()

                        running_loss     += loss.item() * inputs.size(0)
                        running_corrects += torch.sum(preds == labels.data)
                        t.set_postfix(loss=loss.item())

                epoch_loss = running_loss / self.dataset_sizes[phase]
                epoch_acc  = (running_corrects.double() / self.dataset_sizes[phase]).item()

                if phase == "train":
                    self.train_loss_history.append(epoch_loss)
                    self.train_acc_history.append(epoch_acc)
                else:
                    self.val_loss_history.append(epoch_loss)
                    self.val_acc_history.append(epoch_acc)

                print(f"{phase} - Loss: {epoch_loss:.4f}  Acuracia: {epoch_acc:.4f}")

                if phase == "val" and epoch_acc > self.best_acc:
                    self.best_acc = epoch_acc
                    best_wts = copy.deepcopy(self.modelo.state_dict())

        print(f"\nMelhor Acuracia: {self.best_acc:.4f}")
        self.modelo.load_state_dict(best_wts)

    # ── Historico de treinamento ──────────────────────────────────────────

    def plot_training_history(self):
        if not self.train_loss_history:
            return
        e = range(1, len(self.train_loss_history) + 1)
        fig, ax = plt.subplots(1, 2, figsize=(12, 5))
        ax[0].plot(e, self.train_loss_history, label="Treino")
        ax[0].plot(e, self.val_loss_history,   label="Validacao")
        ax[0].set_title("Loss")
        ax[0].legend()
        ax[1].plot(e, self.train_acc_history, label="Treino")
        ax[1].plot(e, self.val_acc_history,   label="Validacao")
        ax[1].set_title("Acuracia")
        ax[1].legend()
        plt.tight_layout()
        plt.show()

    # ── Avaliacao ─────────────────────────────────────────────────────────

    def evaluate(self, plot_confusion_matrix=True, plot_roc=True):
        tl, pl, pp = self._get_predictions(self.dataloaders["val"])
        cm         = confusion_matrix(tl, pl)
        a, b, c, d = cm[0, 0], cm[0, 1], cm[1, 0], cm[1, 1]

        acc   = (a + d) / (a + b + c + d)
        sen   = d / (d + c)         if (d + c) > 0         else 0
        spec  = a / (a + b)         if (a + b) > 0         else 0
        prec  = d / (b + d)         if (b + d) > 0         else 0
        f1    = 2 * prec * sen / (prec + sen) if (prec + sen) > 0 else 0
        denom = math.sqrt((d + b) * (d + c) * (a + b) * (a + c))
        mcc   = (d * a - b * c) / denom if denom > 0 else 0
        auc   = metrics.roc_auc_score(tl, pp) if len(self.class_names) == 2 and pp else None

        print("\nMetricas de Avaliacao:")
        for k, v in [
            ("Acuracia",       acc),
            ("Sensibilidade",  sen),
            ("Especificidade", spec),
            ("Precisao",       prec),
            ("F1-Score",       f1),
            ("MCC",            mcc),
        ]:
            print(f"  {k:20s}: {v:.4f}")
        if auc:
            print(f"  {'AUC-ROC':20s}: {auc:.4f}")

        if plot_confusion_matrix:
            self._plot_confusion_matrix(tl, pl)
        if plot_roc and auc:
            self._plot_roc_curve(f"AUC = {auc:.4f}", tl, pp)

        return {"accuracy": acc, "f1": f1, "mcc": mcc, "auc": auc}

    def _get_predictions(self, dl):
        self.modelo.eval()
        tl, pl = [], []
        pp = [] if len(self.class_names) == 2 else None

        for inp, lbl in dl:
            inp = inp.to(self.device)
            lbl = lbl.to(self.device)
            with torch.no_grad():
                out      = self.modelo(inp)
                _, preds = torch.max(out, 1)
                if pp is not None:
                    probs = torch.nn.functional.softmax(out, dim=1)
                    pp.extend(probs[:, 1].cpu().numpy())
            tl.extend(lbl.cpu().numpy())
            pl.extend(preds.cpu().numpy())
        return tl, pl, pp

    def _plot_confusion_matrix(self, tl, pl):
        cm = confusion_matrix(tl, pl)
        plt.figure(figsize=(7, 6))
        sns.heatmap(
            pd.DataFrame(cm, index=self.class_names, columns=self.class_names),
            annot=True,
            fmt="d",
            cmap="Blues",
        )
        plt.title("Matriz de Confusao", fontsize=14)
        plt.ylabel("Real")
        plt.xlabel("Previsto")
        plt.tight_layout()
        plt.show()

    def _plot_roc_curve(self, name, labels, predictions):
        fp, tp, _ = metrics.roc_curve(labels, predictions)
        plt.figure(figsize=(7, 6))
        plt.plot(fp, tp, label=name, linewidth=2)
        plt.plot([0, 1], [0, 1], "--", color="orange")
        plt.xlabel("Taxa de Falso Positivo")
        plt.ylabel("Taxa de Verdadeiro Positivo")
        plt.title("Curva ROC", fontsize=14)
        plt.legend(loc="lower right")
        plt.tight_layout()
        plt.show()

    # ── Attention Rollout ─────────────────────────────────────────────────

    def visualize_attention_rollout(self, image_path: str):
        """
        Attention Rollout (Abnar & Zuidema, 2020):
        Acumula pesos de atencao de TODAS as camadas e mostra o que o ViT viu.
        """
        tf        = self.data_transforms["val"]
        img_pil   = PILImage.open(image_path).convert("RGB")
        img_array = np.array(img_pil.resize((self.image_size, self.image_size)))
        img_input = tf(img_pil).unsqueeze(0).to(self.device)
        p_sz      = 16 if not self.use_linformer else 32
        n_p       = self.image_size // p_sz

        with torch.no_grad():
            m = self.modelo
            if not hasattr(m, "patch_embed"):
                raise AttributeError("API do ViT nao reconhecida.")

            x = m.patch_embed(img_input)
            b = x.shape[0]
            x = torch.cat([m.cls_token.expand(b, -1, -1), x], dim=1) + m.pos_embed
            if hasattr(m, "pos_drop"):
                x = m.pos_drop(x)

            all_attn = []
            for block in m.blocks:
                B, N, C = x.shape
                am  = block.attn
                nh  = am.num_heads
                hd  = C // nh
                qkv = (
                    am.qkv(block.norm1(x))
                    .reshape(B, N, 3, nh, hd)
                    .permute(2, 0, 3, 1, 4)
                )
                q, k, _ = qkv.unbind(0)
                aw = (q @ k.transpose(-2, -1)) * (hd ** -0.5)
                aw = aw.softmax(dim=-1).mean(dim=1)
                all_attn.append(aw)
                x = block(x)

        rollout = torch.eye(all_attn[0].shape[-1]).unsqueeze(0).to(self.device)
        for aw in all_attn:
            a = aw + torch.eye(aw.shape[-1]).unsqueeze(0).to(self.device)
            a = a / a.sum(dim=-1, keepdim=True)
            rollout = a @ rollout

        cls_attn = rollout[0, 0, 1:].reshape(n_p, n_p).cpu().numpy()
        attn_r   = (
            np.array(
                PILImage.fromarray(
                    (cls_attn / cls_attn.max() * 255).astype(np.uint8)
                ).resize((self.image_size, self.image_size), PILImage.BILINEAR)
            )
            / 255.0
        )

        fig, axes = plt.subplots(1, 3, figsize=(15, 5))
        axes[0].imshow(img_array)
        axes[0].set_title("Imagem original")
        axes[0].axis("off")
        axes[1].imshow(cls_attn, cmap="inferno")
        axes[1].set_title(f"Attention Rollout ({n_p}x{n_p} patches)")
        axes[1].axis("off")
        axes[2].imshow(img_array)
        axes[2].imshow(attn_r, cmap="inferno", alpha=0.6)
        axes[2].set_title("Atencao sobreposta")
        axes[2].axis("off")
        plt.suptitle("Attention Rollout - O que o ViT viu para classificar?", fontsize=14)
        plt.tight_layout()
        plt.show()


print("Classe VisionClassifier definida!")

### 7.2 Treinamento e Avaliação


In [ ]:
# Instanciar o classificador ViT
clf = VisionClassifier(
    data_dir="/content/fiap-ml-visao-computacional/aula-5-machine-learning-aplicado/dados",
    use_linformer=False,
    batch_size=32,
    val_split=0.3,
)

In [ ]:
# Visualizar amostras do dataset de treino
clf.visualize_batch()


In [ ]:
# Treinar por 10 épocas
clf.train(num_epochs=10)


In [ ]:
# Curvas de aprendizado
clf.plot_training_history()


### 7.3 Attention Rollout — Interpretando o ViT

O **Attention Rollout** ([Abnar & Zuidema, 2020](https://arxiv.org/abs/2005.00928)) acumula os pesos de atenção de **todas as camadas** do Transformer.

Mostra quais **patches** da imagem foram mais importantes para a classificação final:
- **Quente (amarelo/branco)**: alta atenção — o modelo focou aqui
- **Frio (preto/roxo)**: baixa atenção — o modelo ignorou

> **Por que isso importa para o negócio?**
> Em diagnósticos médicos, confirma que o modelo olhou para a região correta (lesão pulmonar), não para artefatos ou marcações da imagem.


In [ ]:
# Visualizar mapa de atencao para uma radiografia de tuberculose
clf.visualize_attention_rollout(
    "/content/fiap-ml-visao-computacional/aula-5-machine-learning-aplicado/dados/Tuberculosis/Tuberculosis-10.png"
)

### 7.4 Métricas de Avaliação — Classificação Binária

#### Guia das métricas

| Métrica | Fórmula | Quando priorizar |
|---------|---------|-----------------|
| **Acurácia** | (VP+VN)/Total | Classes balanceadas |
| **Sensibilidade** | VP/(VP+FN) | Minimizar falsos negativos (diagnóstico médico) |
| **Especificidade** | VN/(VN+FP) | Minimizar alarmes falsos |
| **F1-Score** | 2·P·R/(P+R) | Balancear Precisão e Recall |
| **AUC-ROC** | Área sob curva ROC | Comparar modelos independente de threshold |

#### Coeficiente de Correlação de Matthews (MCC)

Considerada a **métrica mais robusta** para classificação binária:

$$MCC = \frac{VP \times VN - FP \times FN}{\sqrt{(VP+FP)(VP+FN)(VN+FP)(VN+FN)}}$$

| Valor | Interpretação |
|-------|--------------|
| **+1** | Predição perfeita |
| **0** | Não melhor que aleatório |
| **-1** | Discordância total |

> O MCC funciona bem mesmo com **classes desbalanceadas** — muito comum em problemas reais de diagnóstico.


In [ ]:
# Avaliação completa do modelo
metricas = clf.evaluate()


---
# Conclusão e Mapa do Conhecimento

## Jornada do Pixel ao Insight

```
 Calibração de Câmera
 Garantir que pixels medem o mundo com precisão
 ↓
 Estimativa de Profundidade (MiDaS)
 Reconstruir dimensão 3D a partir de imagem 2D
 ↓
 CNN do Zero (Keras)
 Aprender features hierárquicas: bordas → formas → objetos
 ↓
 Transfer Learning (VGG19)
 Reutilizar conhecimento de modelos treinados em milhões de imagens
 ↓
 Detecção de Objetos (YOLO12)
 Localizar e classificar múltiplos objetos em tempo real
 ↓
 Segmentação (RF-DETR + SAM)
 Delimitar contornos exatos pixel a pixel
 ↓
 Vision Transformer (ViT + Attention Rollout)
 Atenção global + interpretabilidade para domínios críticos
```

## Quando usar cada abordagem?

| Cenário | Solução Recomendada |
|---------|---------------------|
| Dataset pequeno + GPU limitada | Transfer Learning (MobileNet, EfficientNet) |
| Detecção em tempo real | YOLO (nano/small para edge, large para servidor) |
| Análise detalhada de instâncias | SAM + RF-DETR |
| Dataset grande + relações globais | Vision Transformer (ViT) |
| Domínio crítico (medicina, jurídico) | ViT + Attention Rollout para explicabilidade |
| Deploy em produção | Exportar para ONNX → TensorRT / OpenCV DNN |

## Próximos Passos

1. **Experimente** com seus próprios dados — o código está aqui para ser adaptado
2. **Ajuste hiperparâmetros**: learning rate, batch size, arquitetura, data augmentation
3. **Monitore experimentos**: use [MLflow](https://mlflow.org/) ou [Weights & Biases](https://wandb.ai/)
4. **Implante**: pipeline de CI/CD para modelos de ML com validação antes do deploy
5. **Monitore drift**: dados e conceitos mudam — monitore a performance em produção

---

<div style="background: linear-gradient(135deg, #0f3460, #16213e); padding: 25px; border-radius: 10px; text-align: center; margin-top: 20px;">
<p style="color: #e94560; font-size: 1.4em; margin-bottom: 8px;"> Regra de Ouro</p>
<p style="color: #caf0f8; font-size: 1.2em;"><strong>O modelo é tão bom quanto os dados que recebe.</strong></p>
<p style="color: #8ecae6; font-style: italic;">Garbage in → Garbage out</p>
<p style="color: #a8dadc;">Invista em qualidade dos dados antes de investir em arquiteturas mais complexas.</p>
</div>
